<a href="https://colab.research.google.com/github/Pallavi20004/Demo/blob/main/Ensemble_Techniques.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# Step 1: Load Dataset
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

# Step 2: Data Preprocessing
# Fill missing values
imputer = SimpleImputer(strategy="median")
df['Age'] = imputer.fit_transform(df[['Age']])
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

# Drop unnecessary columns
df.drop(columns=['Cabin', 'Ticket', 'Name', 'PassengerId'], inplace=True)

# Step 3: Feature Engineering
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1  # New Feature

df.drop(columns=['SibSp', 'Parch'], inplace=True)  # Drop original columns

# Step 4: Encode Categorical Features
df = pd.get_dummies(df, columns=['Sex', 'Embarked'], drop_first=True)

# Step 5: Standardize Continuous Features
scaler = StandardScaler()
df[['Fare', 'Age']] = scaler.fit_transform(df[['Fare', 'Age']])

# Step 6: Define Features (X) and Target (y)
X = df.drop(columns=['Survived'])
y = df['Survived']

# Step 7: Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 8: Define Base Models
rf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42)
ada = AdaBoostClassifier(n_estimators=200, learning_rate=0.8, random_state=42)
gb = GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, max_depth=5, random_state=42)

# Step 9: Implement Voting Classifier (Soft Voting)
voting_clf = VotingClassifier(
    estimators=[('rf', rf), ('ada', ada), ('gb', gb)], voting='soft'
)

# Step 10: Train Models
rf.fit(X_train, y_train)
ada.fit(X_train, y_train)
gb.fit(X_train, y_train)
voting_clf.fit(X_train, y_train)

# Step 11: Make Predictions
rf_pred = rf.predict(X_test)
ada_pred = ada.predict(X_test)
gb_pred = gb.predict(X_test)
voting_pred = voting_clf.predict(X_test)

# Step 12: Evaluate Performance
print("\n Model Performance:")
print("Random Forest Accuracy:", accuracy_score(y_test, rf_pred))
print("AdaBoost Accuracy:", accuracy_score(y_test, ada_pred))
print("Gradient Boosting Accuracy:", accuracy_score(y_test, gb_pred))
print("Voting Classifier Accuracy:", accuracy_score(y_test, voting_pred))

# Step 13: Detailed Classification Report
print("\n Voting Classifier Performance Report:")
print(classification_report(y_test, voting_pred))




 Model Performance:
Random Forest Accuracy: 0.8324022346368715
AdaBoost Accuracy: 0.8044692737430168
Gradient Boosting Accuracy: 0.8156424581005587
Voting Classifier Accuracy: 0.8324022346368715

 Voting Classifier Performance Report:
              precision    recall  f1-score   support

           0       0.84      0.89      0.86       105
           1       0.82      0.76      0.79        74

    accuracy                           0.83       179
   macro avg       0.83      0.82      0.82       179
weighted avg       0.83      0.83      0.83       179

